<style>
.jp-Notebook, .jp-Cell, body.jp-Notebook, .cell, div#notebook, .notebook {
  background-color: #0f1115 !important;
}
.jp-RenderedMarkdown, .rendered_html, .jp-RenderedHTMLCommon {
  color: #e6e6e6 !important; font-size: 16px; line-height: 1.6;
}
.jp-RenderedMarkdown h1, .jp-RenderedMarkdown h2, .jp-RenderedMarkdown h3,
.rendered_html h1, .rendered_html h2, .rendered_html h3 { color: #8ec7ff !important; }
.jp-RenderedMarkdown h1, .rendered_html h1 {
  border-bottom: 2px solid #2b4a6f; padding-bottom: .2em;
}
.jp-RenderedMarkdown a, .rendered_html a { color: #7fd1c1 !important; }
.jp-RenderedMarkdown code, .rendered_html code {
  background: #1b2430 !important; color: #ffd479 !important;
  padding: 1px 5px; border-radius: 4px;
}
.jp-RenderedMarkdown pre, .rendered_html pre {
  background: #1b2430 !important; color: #e6e6e6 !important;
  border-left: 3px solid #2b4a6f; padding: .6em .9em; border-radius: 6px;
}
.jp-RenderedMarkdown blockquote, .rendered_html blockquote {
  border-left: 4px solid #7fd1c1; color: #bcd; background: #141a22;
  padding: .3em 1em; border-radius: 0 6px 6px 0;
}
.jp-RenderedMarkdown table, .rendered_html table { color: #e6e6e6; }
.jp-RenderedMarkdown th, .rendered_html th { background: #1b2430 !important; }
</style>


# NB27 · Python de verdad (8): NumPy a fondo, tests y Git

**Parte 3 · Python de verdad — Lección 8 (cierre del bloque)**

> Última lección del bloque de Python. Ya sabes manejar texto, colecciones, errores, funciones, clases, ficheros, módulos y la terminal.
> Hoy cerramos con las tres herramientas que te faltan para trabajar en un equipo profesional de robótica.

1. **NumPy a fondo.** En el NB15 viste lo básico. Pero el día a día con redes neuronales y simuladores es, sobre todo, **manejar arrays**: sus
   tipos, sus formas, sus ejes... y evitar sus trampas. Lo veremos todo, y lo usaremos para simular **mil palos de escoba a la vez**, que es
   exactamente la idea con la que se entrenan los humanoides de verdad.
2. **Tests.** Programas que **comprueban** que tu código funciona, para que un cambio de hoy no rompa en silencio lo que funcionaba ayer.
3. **Git.** La herramienta con la que **todo** el mundo guarda el historial de su código y trabaja en equipo. Este mismo curso vive en Git.

Como en el NB26, las prácticas con ficheros irán a una carpeta `practica_nb27`.


## 1 · El tipo de los números: `dtype`

Todos los números de un array de NumPy son **del mismo tipo**, y ese tipo se llama su **`dtype`** (*data type*). Los dos más importantes en robótica:

In [1]:
import numpy as np

a = np.array([1.5, 2.0, 3.25])
b = a.astype(np.float32)          # convertir a otro tipo
print(a.dtype, "->", a.nbytes, "bytes")
print(b.dtype, "->", b.nbytes, "bytes")

float64 -> 24 bytes
float32 -> 12 bytes


- **`float64`** (el normal): cada número ocupa 8 bytes y tiene unas 16 cifras de precisión. Es lo que da NumPy por defecto.
- **`float32`**: cada número ocupa **la mitad** (4 bytes) y tiene unas 7 cifras de precisión.

¿Por qué importa? Porque las **tarjetas gráficas** (con las que se entrenan las redes y los robots en paralelo) trabajan casi siempre en **`float32`**: la mitad de
memoria y muchísimo más rápido, y 7 cifras de precisión son más que suficientes para una red neuronal. Por eso los espacios de Gymnasium (NB25) piden `float32`, y
por eso las redes de PyTorch usan `float32` por defecto. **Mezclar tipos** es una fuente clásica de errores y avisos (por ejemplo, Gymnasium avisa si tu entorno
devuelve `float64` cuando su espacio dice `float32`).

`astype` convierte un array a otro tipo (devolviendo uno **nuevo**). Y cuidado con los enteros: un array de enteros (`dtype` `int64`) **corta** los decimales si
metes un decimal en él.


## 2 · Fábricas de arrays

Además de `np.array`, `np.zeros` y `np.linspace` (NB15, NB19), hay muchas formas de fabricar arrays:

In [2]:
print(np.arange(0, 1, 0.25))         # como range, pero admite decimales
print(np.ones(3))                     # todo unos
print(np.full(3, 0.4))                # todo un valor
print(np.eye(3))                      # la matriz IDENTIDAD: unos en la diagonal

generador = np.random.default_rng(0)
print(generador.normal(0, 1, size=4).round(2))   # números al azar "en campana" (la veremos en la parte de probabilidad)

[0.   0.25 0.5  0.75]
[1. 1. 1.]
[0.4 0.4 0.4]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[ 0.13 -0.13  0.64  0.1 ]


La **matriz identidad** (`np.eye`) es la que, multiplicada por un vector, lo deja **igual** (como multiplicar un número por 1). Aparece mucho con las rotaciones (en la parte de física).


## 3 · La forma: `reshape`, `ravel` y la traspuesta

Un array tiene un número de **dimensiones** (`ndim`), una **forma** (`shape`) y un número total de elementos (`size`). Y se puede **reorganizar** en otra forma con el mismo
número de elementos, con **`reshape`**:


In [3]:
x = np.arange(12)
print(x, "| ndim:", x.ndim, "| shape:", x.shape)

tabla = x.reshape(3, 4)               # los mismos 12 números, en 3 filas y 4 columnas
print(tabla)
print("ndim:", tabla.ndim, "| shape:", tabla.shape, "| size:", tabla.size)

[ 0  1  2  3  4  5  6  7  8  9 10 11] | ndim: 1 | shape: (12,)
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
ndim: 2 | shape: (3, 4) | size: 12


Un truco muy usado: poner **`-1`** en una de las dimensiones significa "**calcúlala tú**":

In [4]:
print(x.reshape(-1, 6).shape)         # 6 columnas, y las filas que salgan (2)
print(tabla.ravel())                  # de vuelta a una sola fila ("aplanar")
print(tabla.T)                        # la TRASPUESTA: las filas pasan a ser columnas
print(tabla.T.shape)

(2, 6)
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
(4, 3)


La **traspuesta** (`.T`) "gira" la tabla: lo que eran filas son columnas y al revés, así que una forma (3, 4) pasa a (4, 3). Aparece constantemente en las fórmulas de las redes
neuronales (la retropropagación del NB18-19, escrita con matrices, está llena de `.T`).

¿Y si pides una forma imposible?


In [5]:
x.reshape(5, 3)

ValueError: cannot reshape array of size 12 into shape (5,3)

`ValueError: cannot reshape array of size 12 into shape (5,3)`: no se pueden meter 12 números en 5 × 3 = 15 huecos. El tamaño total tiene que coincidir.


## 4 · Índices en dos dimensiones

En una tabla, `M[fila, columna]` (NB15). Y con porciones (`:`) en cada dimensión se cogen filas enteras, columnas enteras o trozos:

In [6]:
print(tabla)
print("fila 1:       ", tabla[1])
print("columna 2:    ", tabla[:, 2])          # ":" = todas las filas; 2 = la columna 2
print("trozo:\n", tabla[0:2, 1:3])           # filas 0-1, columnas 1-2
print("última columna:", tabla[:, -1])

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
fila 1:        [4 5 6 7]
columna 2:     [ 2  6 10]
trozo:
 [[1 2]
 [5 6]]
última columna: [ 3  7 11]


`tabla[:, 2]` se lee "**todas** las filas, columna 2": la columna entera. Es la forma de sacar, por ejemplo, **todas las inclinaciones** de una tabla de observaciones donde cada fila
es un paso y cada columna un dato.


## 5 · Máscaras: filtrar con condiciones

Una comparación con un array da **un array de `True` y `False`**, uno por elemento (lo viste en el NB19, con `zs > 0`). Se llama **máscara**, y se puede usar **como índice** para quedarse
solo con los elementos donde vale `True`:


In [7]:
retornos = np.array([44.8, 499.9, 168.5, 499.7, 296.0, 43.2])

buenos = retornos > 400
print(buenos)
print(retornos[buenos])                       # solo los que cumplen
print("¿Cuántos?", buenos.sum(), "| proporción:", buenos.mean())

[False  True False  True False False]
[499.9 499.7]
¿Cuántos? 2 | proporción: 0.3333333333333333


`retornos[retornos > 400]` filtra en una línea, sin bucle ni comprensión. Y como `True` vale 1 y `False` vale 0, **`.sum()`** cuenta los que cumplen, y **`.mean()`** da la **proporción**
(aquí, 2 de 6). Las máscaras se combinan con **`&`** ("y"), **`|`** ("o") y **`~`** ("no"), siempre con paréntesis: `(retornos > 100) & (retornos < 400)`.

Para **elegir** entre dos valores según una condición, elemento a elemento, **`np.where(condición, si_cierto, si_falso)`** (lo viste en el ejercicio E6 del NB19):


In [8]:
print(np.where(retornos > 400, "completo", "caído"))

['caído' 'completo' 'caído' 'completo' 'caído' 'caído']


Y para saber **dónde** está el mayor o en qué orden van los elementos: **`argmax`** (la **posición** del mayor) y **`argsort`** (las posiciones que **ordenarían** el array):

In [9]:
print("El mejor está en la posición", retornos.argmax())
orden = retornos.argsort()[::-1]                 # de mayor a menor (NB21: [::-1] da la vuelta)
print("Posiciones de mejor a peor:", orden)
print("Retornos ordenados:        ", retornos[orden])

El mejor está en la posición 1
Posiciones de mejor a peor: [1 3 4 2 0 5]
Retornos ordenados:         [499.9 499.7 296.  168.5  44.8  43.2]


¿Has visto `retornos[orden]`? Un array se puede usar como índice con **una lista de posiciones**, y devuelve esos elementos en ese orden. Muy útil: si tienes los retornos y las
políticas en dos arrays paralelos, `politicas[retornos.argsort()]` ordena las políticas por su retorno.


## 6 · Los ejes: hacer cuentas por filas o por columnas

Esta es, quizá, la idea más importante de NumPy para el trabajo diario. Imagina una tabla de resultados: cada **fila** es una política, cada **columna** una semilla (los retornos del
NB11):


In [10]:
resultados = np.array([
    [41.6,  46.2,  43.1,  45.3,  47.6],      # nada
    [499.7, 168.5, 144.5, 198.8, 468.3],     # solo inclinación
    [499.9, 499.9, 499.9, 499.9, 499.9],     # a mano
])
print(resultados.shape)
print("Media de TODO:", resultados.mean().round(1))

(3, 5)
Media de TODO: 280.2


`mean()` a secas da la media de **todos** los números juntos, que no significa gran cosa. Lo que queremos es la media **de cada política** (de cada fila), o **de cada semilla** (de cada
columna). Para eso, las funciones de NumPy aceptan un **eje** (`axis`):


In [11]:
print("Media por política (axis=1):", resultados.mean(axis=1).round(1))
print("Media por semilla  (axis=0):", resultados.mean(axis=0).round(1))
print("Peor episodio de cada política:", resultados.min(axis=1))

Media por política (axis=1): [ 44.8 296.  499.9]
Media por semilla  (axis=0): [347.1 238.2 229.2 248.  338.6]
Peor episodio de cada política: [ 41.6 144.5 499.9]


La regla para no liarse: **el eje que dices es el que desaparece**. La tabla es (3 políticas, 5 semillas):

```
                      ← axis=1 (a lo largo de las semillas, "hacia la derecha") →
                    semilla 0  semilla 1  semilla 2  semilla 3  semilla 4
   ↑   nada            41.6      46.2      43.1      45.3      47.6     → mean(axis=1) = 44.8
 axis=0 inclinación   499.7     168.5     144.5     198.8     468.3     → mean(axis=1) = 296.0
   ↓   a mano         499.9     499.9     499.9     499.9     499.9     → mean(axis=1) = 499.9
                        ↓         ↓         ↓         ↓         ↓
                    mean(axis=0): una media por semilla (5 números)
```

- `axis=1` **recorre** las columnas (las semillas) y las **resume**: queda un número por fila → forma (3,).
- `axis=0` **recorre** las filas (las políticas) y las resume: queda un número por columna → forma (5,).

Funciona igual con `sum`, `max`, `min`, `std` (la desviación típica), `argmax`... **Siempre que una cuenta te salga con la forma equivocada, revisa el `axis`.**


## 7 · Broadcasting: operar con formas distintas

En el NB15 viste que sumar arrays de formas **distintas** daba un `ValueError` con la palabra *broadcast*. Pero a veces NumPy **sí** combina formas distintas, y con mucha inteligencia: estira
la más pequeña para que encaje. Se llama **broadcasting** ("difusión"). El caso más simple ya lo conoces: `2 * array` "estira" el 2 a todo el array.

El caso más útil en robótica: **normalizar las observaciones**. Las redes neuronales aprenden mucho mejor si cada dato de la observación está "centrado en 0" y con un tamaño parecido (la
inclinación del palo va de −30 a 30, pero su velocidad puede ir de −100 a 100: muy distintos). Se hace restando a **cada columna** su media y dividiendo entre su desviación típica:


In [12]:
generador = np.random.default_rng(1)
observaciones = np.column_stack([generador.normal(5, 2, size=6),        # columna 0: inclinaciones
                                 generador.normal(-40, 30, size=6)])    # columna 1: velocidades
print("Forma:", observaciones.shape)

media = observaciones.mean(axis=0)            # una media por columna: forma (2,)
desviacion = observaciones.std(axis=0)        # una desviación por columna: forma (2,)
normalizadas = (observaciones - media) / desviacion    # ¡(6, 2) menos (2,)!

print("Media de cada columna, antes:  ", media.round(2))
print("Media de cada columna, después:", normalizadas.mean(axis=0).round(2))
print("Desviación, después:           ", normalizadas.std(axis=0).round(2))

Forma: (6, 2)
Media de cada columna, antes:   [  5.52 -33.61]
Media de cada columna, después: [-0. -0.]
Desviación, después:            [1. 1.]


(`np.column_stack` pega varios arrays como columnas de una tabla.) La línea clave es `observaciones - media`: una tabla de forma **(6, 2)** menos un vector de forma **(2,)**. NumPy "repite"
el vector en cada una de las 6 filas, así que a cada fila se le resta la media de **su** columna. Resultado: cada columna, centrada en 0 y con desviación 1. **Esto se hace de verdad** al
entrenar robots (lo verás como "normalización de observaciones").

Las **reglas** del broadcasting: NumPy compara las formas **empezando por la derecha**; cada pareja de tamaños tiene que ser **igual**, o **uno de ellos ser 1** (o no existir). (6, 2) y (2,):
a la derecha, 2 y 2, iguales; a la izquierda, 6 y "nada", vale. Si no se cumple:


In [13]:
observaciones - np.array([1.0, 2.0, 3.0])

ValueError: operands could not be broadcast together with shapes (6,2) (3,) 

(6, 2) y (3,): a la derecha, 2 contra 3, ni iguales ni 1. Error. **El broadcasting es potentísimo, pero también peligroso**: a veces encaja formas que **no** querías combinar, y el
resultado es un array de forma inesperada **sin ningún error**. Por eso, la costumbre profesional del NB22: **mirar siempre `.shape`**.


## 8 · Apilar arrays

Para juntar arrays: **`np.stack`** los apila creando una dimensión **nueva**, y **`np.concatenate`** los pega a lo largo de una dimensión **existente**:

In [14]:
paso1 = np.array([2.0, 0.0])
paso2 = np.array([2.02, 0.81])
paso3 = np.array([2.05, 1.55])

trayectoria = np.stack([paso1, paso2, paso3])         # 3 observaciones de 2 -> tabla (3, 2)
print(trayectoria.shape)
print(np.concatenate([paso1, paso2]))                # pegadas en fila -> (4,)

(3, 2)
[2.   0.   2.02 0.81]


`np.stack` es lo que se usa para convertir una **lista de observaciones** (las que vas guardando paso a paso, NB09) en una **tabla**, una fila por paso. Así se preparan los datos para
entrenar.


## 9 · La trampa del alias en NumPy: vistas

En el NB21 viste la trampa del alias con las listas. NumPy tiene **su propia versión**, más sutil. Una **porción** de un array **no es una copia**: es una **vista**, una "ventana" a los mismos
datos. Si cambias la porción, **cambias el original**:


In [15]:
pesos = np.zeros(6)
primeros = pesos[0:3]          # ¡una VISTA, no una copia!
primeros[0] = 99
print(pesos)

[99.  0.  0.  0.  0.  0.]


¡El 99 aparece en `pesos`! (Con una lista normal, `lista[0:3]` **sí** sería una copia; NumPy funciona distinto a propósito, para no gastar memoria copiando arrays enormes.) Si necesitas
una copia independiente, **`.copy()`**:


In [16]:
pesos = np.zeros(6)
primeros = pesos[0:3].copy()
primeros[0] = 99
print(pesos)

[0. 0. 0. 0. 0. 0.]


Ahora `pesos` sigue intacto. **Dónde muerde**: guardas los pesos "buenos" de una red con `mejores = pesos[...]` para recuperarlos luego, sigues entrenando... y los "buenos" han cambiado con
el entrenamiento, porque eran una vista. **Al guardar un estado para después, siempre `.copy()`.** (Las máscaras y las listas de posiciones del apartado 5, en cambio, sí devuelven copias.)


## 10 · Proyecto: mil palos de escoba a la vez

Y ahora, la aplicación más potente de todo lo anterior. Hasta ahora simulábamos los episodios **de uno en uno**, con bucles de Python. Pero ¿y si simulamos **mil palos a la vez**? En vez de una
inclinación, un **array** de mil inclinaciones; en vez de un viento, un array de mil vientos. Y la física se aplica a **todos a la vez**, con operaciones de NumPy. Se llama **vectorizar**.

La única dificultad: cada palo se cae en un momento distinto. Lo resolvemos con una **máscara** de "vivos": los caídos dejan de sumar recompensa. Léelo despacio; cada línea es algo de hoy:


In [17]:
def evaluar_en_paralelo(k, d, n_palos=1000, semilla=0, viento_maximo=30):
    generador = np.random.default_rng(semilla)
    inclinacion = np.full(n_palos, 2.0)          # mil inclinaciones (apartado 2)
    velocidad = np.zeros(n_palos)
    vivos = np.ones(n_palos, dtype=bool)         # una máscara: todos empiezan vivos
    retornos = np.zeros(n_palos)

    for paso in range(500):
        empuje = np.clip(-k * inclinacion - d * velocidad, -40, 40)       # la política, para los mil
        viento = generador.uniform(-viento_maximo, viento_maximo, size=n_palos)
        aceleracion = 10 * inclinacion + empuje + viento                   # la física, para los mil
        velocidad = velocidad + aceleracion * 0.02
        inclinacion = inclinacion + velocidad * 0.02
        vivos = vivos & (np.abs(inclinacion) <= 30)                        # los que caen, dejan de estar vivos
        retornos = retornos + np.where(vivos, 1 - (inclinacion / 30) ** 2, 0.0)

    return retornos

import time
for nombre, k, d in [("nada", 0, 0), ("solo inclinación", 30, 0), ("a mano", 30, 8)]:
    inicio = time.perf_counter()
    retornos_mil = evaluar_en_paralelo(k, d)
    duracion = time.perf_counter() - inicio
    caidos = (retornos_mil < 490).mean()
    print(f"{nombre:>17}: media {retornos_mil.mean():6.1f} | peor {retornos_mil.min():6.1f} | "
          f"se cae el {caidos:5.1%} | {duracion * 1000:.0f} ms")

             nada: media   44.9 | peor   38.9 | se cae el 100.0% | 28 ms
 solo inclinación: media  339.2 | peor   79.6 | se cae el 63.8% | 28 ms


           a mano: media  499.9 | peor  499.9 | se cae el  0.0% | 28 ms


**Mil episodios en unas pocas centésimas de segundo** por política. Y los resultados nos enseñan algo importante:

- "a mano": **499,9** de media y de **peor caso**: no se cae **ninguna** de las mil veces.
- "solo inclinación": se cae alrededor de **dos de cada tres** veces (en torno al 63 %). Con nuestras 5 semillas del NB11 parecía un 60 % (3 de 5)... y con otras 5 semillas, en el NB25, salió
  bastante mejor. **Cinco episodios son muy pocos para medir algo con fiabilidad.** Con mil, el número ya es de fiar. (La estadística, en la próxima parte, nos dirá exactamente **cuánto** fiarnos.)

Comparémoslo con la versión de bucles de toda la vida, para la política a mano, con los mismos mil episodios:


In [18]:
import random

def evaluar_con_bucles(k, d, n_episodios=1000):
    total = 0
    for semilla in range(n_episodios):
        azar = random.Random(semilla)
        inclinacion, velocidad = 2.0, 0.0
        for paso in range(500):
            empuje = max(-40, min(40, -k * inclinacion - d * velocidad))
            velocidad += (10 * inclinacion + empuje + azar.uniform(-30, 30)) * 0.02
            inclinacion += velocidad * 0.02
            if abs(inclinacion) > 30:
                break
            total += 1 - (inclinacion / 30) ** 2
    return total / n_episodios

inicio = time.perf_counter()
media_bucles = evaluar_con_bucles(30, 8)
duracion_bucles = time.perf_counter() - inicio

inicio = time.perf_counter()
media_paralelo = evaluar_en_paralelo(30, 8).mean()
duracion_paralelo = time.perf_counter() - inicio

print(f"Con bucles:    {media_bucles:.1f} en {duracion_bucles * 1000:.0f} ms")
print(f"En paralelo:   {media_paralelo:.1f} en {duracion_paralelo * 1000:.0f} ms")
print(f"Unas {duracion_bucles / duracion_paralelo:.0f} veces más rápido")

Con bucles:    499.9 en 377 ms
En paralelo:   499.9 en 28 ms
Unas 14 veces más rápido


El mismo resultado, **entre diez y veinte veces más rápido** (el número exacto cambia en cada ejecución). Y con más palos la ventaja crece.

**Esta es exactamente la idea de los simuladores profesionales para entrenar humanoides** (lo veremos en la Parte 5): en vez de un robot, simulan **miles a la vez**, con arrays enormes, en la
tarjeta gráfica. Por eso un entrenamiento que llevaría semanas con un robot cada vez se hace en minutos. Acabas de escribir, en pequeño, un **entorno vectorizado**.


## 11 · Tests: programas que comprueban tu código

Imagina que cambias una línea de la física del palo de escoba para "mejorarla"... y sin darte cuenta rompes algo. Todo sigue funcionando, no hay errores, pero los resultados ya no son correctos.
Puedes tardar días en darte cuenta. Los **tests** (o **pruebas**) evitan esto: son pequeñas funciones que **comprueban automáticamente** que tu código hace lo que debe. Cada vez que cambias algo,
los ejecutas, y si algo se ha roto, **te lo dicen al instante**.

Un test es, en el fondo, un **`assert`** (NB22) dentro de una función:

```python
def test_reset_empieza_inclinado():
    entorno = PaloDeEscoba()
    (inclinacion, velocidad), info = entorno.reset(seed=0)
    assert inclinacion == 2.0
    assert velocidad == 0.0
```

La herramienta estándar para ejecutar tests en Python se llama **pytest**. Su funcionamiento es sencillísimo: busca ficheros cuyo nombre empiece por `test_`, dentro de ellos las funciones que
empiecen por `test_`, las ejecuta todas, y te dice cuáles pasan y cuáles fallan.

Vamos a escribir un fichero de tests para el módulo `palo.py` del NB26 (lo copiamos a nuestra carpeta de prácticas):


In [19]:
from pathlib import Path
import shutil

carpeta = Path("practica_nb27")
carpeta.mkdir(exist_ok=True)
origen = Path("practica_nb26") / "palo.py"
if not origen.exists():
    raise FileNotFoundError("Ejecuta primero el NB26: este apartado usa el módulo palo.py que se crea allí.")
shutil.copy(origen, carpeta / "palo.py")          # shutil: copiar ficheros (biblioteca estándar)
print("palo.py copiado")

palo.py copiado


(Fíjate en el `raise` con un mensaje claro si falta el fichero: fallar pronto, NB22.) Y ahora, el fichero de tests:

In [20]:
tests = '''# test_palo.py: tests del entorno del palo de escoba.
import pytest
from palo import PaloDeEscoba, evaluar


def test_reset_empieza_inclinado():
    entorno = PaloDeEscoba()
    (inclinacion, velocidad), info = entorno.reset(seed=0)
    assert inclinacion == 2.0
    assert velocidad == 0.0


def test_misma_semilla_mismo_resultado():
    # reproducibilidad: dos evaluaciones con las mismas semillas dan lo mismo
    assert evaluar(30, 8) == evaluar(30, 8)


def test_recompensa_entre_0_y_1():
    entorno = PaloDeEscoba()
    entorno.reset(seed=3)
    for paso in range(100):
        obs, recompensa, terminado, truncado, info = entorno.step(0)
        assert 0.0 <= recompensa <= 1.0
        if terminado:
            break


def test_no_hacer_nada_se_cae():
    assert evaluar(0, 0) < 100


def test_politica_a_mano_aguanta():
    assert evaluar(30, 8) == pytest.approx(499.93, abs=0.05)
'''
(carpeta / "test_palo.py").write_text(tests, encoding="utf-8")
print("Tests escritos")

Tests escritos


Cinco tests, cada uno comprueba **una** cosa, con un nombre que dice **qué**. Fíjate en el último: **`pytest.approx`**. Comparar decimales con `==` es peligroso (el ruido de los decimales del
NB06: `0.1 + 0.2 == 0.3` es `False`). `pytest.approx(valor, abs=0.05)` significa "aproximadamente este valor, con un margen de 0,05". (NumPy tiene su equivalente para arrays:
`np.testing.assert_allclose`.)

Ejecutémoslos. En una terminal escribirías `pytest` en la carpeta; desde aquí, con `subprocess` (NB26):


In [21]:
import subprocess, sys

def ejecutar_tests():
    resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"],
                               cwd=carpeta, capture_output=True, text=True)
    print(resultado.stdout.strip()[-600:])

ejecutar_tests()

.....                                                                    [100%]
5 passed in 0.02s


**`5 passed`**: los cinco tests pasan. (Cada punto `.` es un test que ha ido bien. `-q` pide un resumen corto, y `-p no:cacheprovider` evita que pytest deje una carpeta de caché.)

Ahora, lo importante: **¿cazan los tests un fallo?** Vamos a "estropear" `palo.py` a propósito, como si alguien hubiera cambiado sin querer la gravedad de 10 a 1 (un error de una sola tecla), y
volvemos a ejecutar los tests:


In [22]:
ruta_palo = carpeta / "palo.py"
original = ruta_palo.read_text(encoding="utf-8")
_ = ruta_palo.write_text(original.replace("10 * self.inclinacion", "1 * self.inclinacion"), encoding="utf-8")

ejecutar_tests()

_ = ruta_palo.write_text(original, encoding="utf-8")   # lo dejamos como estaba

                                           [100%]
=================================== FAILURES ===================================
__________________________ test_no_hacer_nada_se_cae ___________________________

    def test_no_hacer_nada_se_cae():
>       assert evaluar(0, 0) < 100
E       assert 137.79098583269757 < 100
E        +  where 137.79098583269757 = evaluar(0, 0)

test_palo.py:29: AssertionError
=========================== short test summary info ============================
FAILED test_palo.py::test_no_hacer_nada_se_cae - assert 137.79098583269757 < 100
1 failed, 4 passed in 0.03s


**`1 failed, 4 passed`**: el test `test_no_hacer_nada_se_cae` ha **fallado**. Con una gravedad diez veces más débil, el palo ya no se cae aunque no hagas nada, y el test lo ha detectado
**al instante**, diciendo **qué test** falló y **por qué** (pytest muestra el `assert` que no se cumplió, con los valores). Sin el test, ese error de una tecla podría haber pasado desapercibido
durante semanas.

**Costumbres profesionales con los tests:**

- Cada test comprueba **una** cosa, con un nombre que diga **qué**.
- Se ejecutan **antes de subir cualquier cambio**. En las empresas, se ejecutan solos cada vez que alguien sube código (se llama **integración continua**).
- Cuando encuentras un fallo, **primero escribes un test que lo reproduzca**, y luego lo arreglas: así ese fallo no vuelve jamás.
- En robótica, se prueban cosas como: que el entorno sea **reproducible** con la misma semilla, que las observaciones tengan la **forma** y el **tipo** correctos, que las recompensas estén en su
  rango, y que una política conocida dé el resultado esperado.


## 12 · Git: el historial de tu código

Llevas todo el curso viendo que "este notebook se sube a GitHub". Ahora vas a entender qué es eso.

**Git** es un programa de **control de versiones**: guarda el **historial completo** de un proyecto. Cada vez que llegas a un punto que merece la pena (un notebook terminado, un fallo arreglado),
haces una **"foto"** de todos los ficheros, con un mensaje que explica qué cambiaste. Esa foto se llama **commit** ("confirmación"). Con Git puedes:

- **Volver atrás** a cualquier foto anterior (si rompes algo, recuperas la versión buena).
- **Ver qué cambió** entre dos fotos, línea a línea (el **diff**, "diferencia").
- **Trabajar en paralelo** en una **rama** (*branch*): una línea de trabajo separada para probar algo sin tocar lo principal, y **fusionarla** (*merge*) si sale bien.
- **Trabajar en equipo**: una copia del proyecto vive en un servidor **remoto** (como **GitHub**), y cada persona **sube** (*push*) y **baja** (*pull*) los cambios.

Los conceptos, en una tabla:

| Concepto | Qué es |
|---|---|
| **Repositorio** | La carpeta del proyecto, con todo su historial (en una carpeta oculta `.git`) |
| **Commit** | Una "foto" de los ficheros, con un mensaje, una fecha y un autor |
| **Área de preparación** (*staging*) | Donde eliges qué cambios van en el próximo commit (`git add`) |
| **Rama** (*branch*) | Una línea de trabajo; la principal suele llamarse `main` |
| **Remoto** | Una copia del repositorio en otro sitio (GitHub) |
| **`push` / `pull`** | Subir tus commits al remoto / bajar los de los demás |

Vamos a practicar con un **repositorio de juguete** dentro de nuestra carpeta de prácticas. Una pequeña función para lanzar órdenes de Git (con `subprocess`, NB26):


In [23]:
repo = carpeta / "mi_robot"
if repo.exists():
    shutil.rmtree(repo)               # empezamos de cero cada vez (rmtree borra una carpeta entera)
repo.mkdir()

def git(*ordenes):
    resultado = subprocess.run(["git", *ordenes], cwd=repo, capture_output=True, text=True)
    salida = (resultado.stdout + resultado.stderr).strip()
    if salida:
        print(salida)

git("init", "-q", "-b", "main")                             # crear el repositorio, con la rama 'main'
git("config", "user.name", "Alumno de robótica")           # quién firma los commits (solo en este repositorio)
git("config", "user.email", "alumno@ejemplo.com")

`git init` ha convertido la carpeta en un repositorio (vacío; `-q` le pide que no diga nada). `git config` dice quién firma los commits (cada persona lo configura una vez en su ordenador; aquí lo hacemos solo para este
repositorio de juguete). Ahora creamos un fichero, lo **preparamos** con `git add` y hacemos el **primer commit**:


In [24]:
(repo / "politica.py").write_text("K = 30\nD = 8\n", encoding="utf-8")

git("add", "politica.py")                                   # 1. preparar el fichero para la foto
git("commit", "-q", "-m", "Primera política: k=30, d=8")   # 2. hacer la foto, con su mensaje
git("log", "--oneline")                                     # 3. ver el historial

8f5b155 Primera política: k=30, d=8


El historial ya tiene **un commit**, identificado por un código (los primeros caracteres de un número larguísimo, único para cada commit) y su mensaje. Cambiemos el fichero y veamos qué ha
cambiado **antes** de hacer la siguiente foto, con `git diff`:


In [25]:
(repo / "politica.py").write_text("K = 25\nD = 6\n", encoding="utf-8")
git("diff")

diff --git a/politica.py b/politica.py
index 31bc02f..6e2e03c 100644
--- a/politica.py
+++ b/politica.py
@@ -1,2 +1,2 @@
-K = 30
-D = 8
+K = 25
+D = 6


El **diff** muestra, línea a línea, lo que ha cambiado: las líneas con **`-`** delante son las que se han quitado y las de **`+`**, las que se han añadido. Las ruedecillas pasaron de 30 y 8 a 25 y
6. Es la herramienta que usan los equipos para **revisar** los cambios de los demás antes de aceptarlos. Hagamos el segundo commit:


In [26]:
git("commit", "-q", "-am", "Ruedecillas aprendidas por el gradiente (NB17): k=25, d=6")
git("log", "--oneline")

8f180a9 Ruedecillas aprendidas por el gradiente (NB17): k=25, d=6
8f5b155 Primera política: k=30, d=8


(`-am` = añadir todos los ficheros ya conocidos que hayan cambiado, y hacer el commit, en un paso.) Dos fotos en el historial, de la más reciente a la más antigua.

Ahora, una **rama**: queremos **probar** una idea arriesgada (unas ruedecillas enormes) **sin tocar** la versión buena. Creamos una rama, cambiamos el fichero allí, y hacemos un commit en ella:


In [27]:
git("switch", "-q", "-c", "prueba-ruedecillas-grandes")    # crear una rama nueva y cambiarse a ella
(repo / "politica.py").write_text("K = 80\nD = 20\n", encoding="utf-8")
git("commit", "-q", "-am", "Probar ruedecillas grandes")

git("switch", "-q", "main")                                 # volver a la rama principal
print("En main, el fichero dice:", (repo / "politica.py").read_text(encoding="utf-8").split())
git("log", "--oneline", "--all", "--decorate")

En main, el fichero dice: ['K', '=', '25', 'D', '=', '6']


8f180a9 (HEAD -> main) Ruedecillas aprendidas por el gradiente (NB17): k=25, d=6
dbd5959 (prueba-ruedecillas-grandes) Probar ruedecillas grandes
8f5b155 Primera política: k=30, d=8


Al volver a `main`, el fichero vuelve a tener **25 y 6**: la prueba vive **solo** en su rama. El historial (con `--all`, que muestra todas las ramas, y `--decorate`,
que pone las etiquetas) lo enseña: el commit de la prueba lleva la etiqueta de su rama, `prueba-ruedecillas-grandes`, y `main` (con `HEAD`, "donde estás ahora") sigue en el commit
anterior. Si la prueba saliera bien, se **fusionaría** con
`git merge prueba-ruedecillas-grandes`; si sale mal, se abandona la rama, y `main` nunca se enteró. Así trabajan los equipos: cada idea, en su rama; a `main`, solo lo que funciona.

### El `.gitignore`

No todo debe ir al historial. Las carpetas enormes o que se pueden regenerar (como el entorno virtual `venv`), las cachés, y **sobre todo las contraseñas y claves secretas**, se excluyen con un fichero
**`.gitignore`**: una lista de patrones de ficheros que Git debe ignorar. Mira el de este curso:


In [28]:
print(Path("..", ".gitignore").read_text(encoding="utf-8"))

# Entorno virtual (enorme y específico de la máquina; se reinstala con requirements/)
venv/
env/
.venv/

# Cachés de Python
__pycache__/
*.pyc
*.pyo

# Checkpoints de Jupyter
.ipynb_checkpoints/
*/.ipynb_checkpoints/

# Sistema
.DS_Store
Thumbs.db

# Carpetas de prácticas que crean los notebooks al ejecutarse
notebooks/practica_*/
trabajo_nb35/*.zip
trabajo_nb35/*.pkl
trabajo_nb35/parcial_*/
trabajo_nb35/err_*.txt

# Vídeos de las prácticas de MuJoCo (se regeneran al ejecutar; ya van incrustados en el notebook)
notebooks/assets/practicas/



Ahí están `venv/` (el entorno virtual, NB26), las cachés de Python (`__pycache__/`) y las carpetas de prácticas como esta (`notebooks/practica_*/`). **Regla de oro: jamás subas contraseñas, claves de
APIs ni datos privados a Git.** Aunque los borres después, siguen en el historial.

### Este curso, en Git

Este mismo curso es un repositorio de Git, y cada notebook terminado ha sido un commit. Mira los últimos:


In [29]:
historial = subprocess.run(["git", "log", "--oneline", "-5"], cwd="..", capture_output=True, text=True).stdout
print(historial)

30a9daf Plan: Parte 1 completa; nota mj_setConst
519f3e0 Prácticas MuJoCo Parte 1 (NB05-NB11b): print del modelo, variables en opt/body_mass, tu bucle mj_step, if/break, trayectorias y rebote, caida(gravedad), palo de escoba en MJCF con física real, leer un script MuJoCo
e1c0728 Plan: Parte 0 completa
ef18e74 Prácticas MuJoCo NB03, NB03b, NB04, NB04b, NB05b: mente al humanoide (ctrl/observación/4 políticas), unidades y radianes, recompensa real del Humanoid-v5, fórmula de caída vs MuJoCo, MuJoCo desde la terminal
e376672 PROGRESO: nota para retomar las prácticas de MuJoCo



Los últimos commits del curso: cada lección, una foto. Y desde ahí, `git push` los sube a GitHub, de donde los lees en otro dispositivo.

Las órdenes de Git que usarás a diario:

| Orden | Qué hace |
|---|---|
| `git status` | ¿Qué ha cambiado y qué está preparado? (la orden más usada) |
| `git add fichero` | Preparar cambios para el próximo commit |
| `git commit -m "mensaje"` | Hacer la foto |
| `git log --oneline` | Ver el historial |
| `git diff` | Ver los cambios aún no confirmados |
| `git switch -c rama` / `git switch rama` | Crear una rama y cambiarse / cambiarse a una rama |
| `git merge rama` | Traer los cambios de otra rama a la actual |
| `git clone URL` | Descargar un repositorio entero (por ejemplo, de GitHub) |
| `git pull` / `git push` | Bajar / subir cambios del remoto |

Y un consejo sobre los **mensajes**: que digan **qué** y **por qué**, en una línea clara ("Arregla la recompensa: el castigo por esfuerzo tenía el signo cambiado"). Un historial con mensajes como
"cambios" o "arreglo" no sirve de nada dentro de seis meses.


## 13 · Escribir código como un profesional

Para terminar el bloque, las costumbres que distinguen el código profesional. Casi todas ya las has visto a lo largo del curso:

- **Estilo PEP 8**: la guía de estilo oficial de Python. Nombres `en_minusculas_con_guiones` para variables y funciones, `CadaPalabraConMayuscula` para clases, `MAYUSCULAS` para constantes (NB11);
  **4 espacios** de sangría; líneas no demasiado largas; espacios alrededor de `=` y de los operadores. Hay herramientas que lo comprueban y lo arreglan solas.
- **Nombres que expliquen** (NB06): `velocidad_hacia_delante`, no `v`.
- **Funciones pequeñas que hacen una sola cosa**, con docstring (NB23).
- **Nada de números mágicos** (NB06): constantes con nombre o configuración.
- **Fallar pronto** con mensajes claros (NB22).
- **Reproducibilidad** (NB11, NB26): semillas fijas, configuración guardada, versiones apuntadas en `requirements.txt`.
- **Tests** para lo importante, y **Git** para todo.
- **Comentarios que explican el porqué**, no el qué (el código ya dice el qué).
- **Simple antes que ingenioso**: el código se lee muchas más veces de las que se escribe.


## 14 · Resumen de la lección (y del bloque)

1. **NumPy a fondo**: `dtype` (`float32` para tarjetas gráficas; `astype`), fábricas (`arange`, `ones`, `full`, `eye`, `normal`), forma (`reshape` con `-1`, `ravel`, `.T`), índices 2D (`M[:, j]`), **máscaras**
   (`x[x > 400]`, `.sum()`, `.mean()`, `np.where`), `argmax`/`argsort`.
2. **Ejes**: `mean(axis=1)` resume cada fila, `axis=0` cada columna (el eje que dices desaparece). **Broadcasting**: formas distintas que encajan comparando desde la derecha (normalizar observaciones).
   **Vistas**: una porción de un array **no** es una copia; usa `.copy()`.
3. **Vectorizar**: mil palos de escoba a la vez, con máscaras de "vivos", entre diez y veinte veces más rápido; y la lección de que **5 episodios no bastan para medir** ("solo inclinación" se cae ~63 % de las veces).
4. **Tests** con **pytest** (funciones `test_...` con `assert`, `pytest.approx` para decimales): cazan al instante un fallo de una sola tecla.
5. **Git**: commits (fotos con mensaje), `add`/`commit`/`log`/`diff`, ramas (`switch -c`, `merge`), remotos (`push`/`pull`), `.gitignore` (jamás secretos). Y el estilo profesional (PEP 8 y buenas costumbres).

### Palabras nuevas de hoy

| Palabra | Qué significa |
|---|---|
| **`dtype`** | El tipo de los números de un array (`float64`, `float32`, `int64`...). |
| **Traspuesta (`.T`)** | La tabla "girada": filas por columnas. |
| **Matriz identidad** | La que deja igual a un vector al multiplicarlo (`np.eye`). |
| **Máscara** | Array de `True`/`False` para filtrar otro array. |
| **Eje (`axis`)** | La dimensión a lo largo de la que se hace una cuenta. |
| **Broadcasting** | Operar con arrays de formas distintas que NumPy hace encajar. |
| **Normalizar observaciones** | Centrar cada dato en 0 y con tamaño 1, para que la red aprenda mejor. |
| **Vista** | Una porción de un array que comparte los datos con el original. |
| **Vectorizar** | Hacer con arrays, de golpe, lo que se haría con un bucle. |
| **Test / pytest** | Función que comprueba el código / la herramienta para ejecutarlas. |
| **Integración continua** | Ejecutar los tests automáticamente cada vez que se sube código. |
| **Git / repositorio / commit** | Control de versiones / proyecto con historial / una "foto" del proyecto. |
| **Diff** | Las diferencias, línea a línea, entre dos versiones. |
| **Rama / merge** | Una línea de trabajo separada / fusionarla con otra. |
| **Remoto / push / pull** | Copia en un servidor (GitHub) / subir / bajar cambios. |
| **PEP 8** | La guía de estilo oficial de Python. |


## 15 · Ejercicios

**E1.** Crea con `np.arange` y `reshape` una tabla de 4 filas y 5 columnas con los números del 0 al 19. Saca la tercera columna y la última fila.

**E2.** Con la tabla `resultados` del apartado 6, calcula la **desviación típica** de cada política (`std` con el `axis` correcto). ¿Cuál es la más irregular?

**E3.** Predice la forma del resultado: (a) un array (100, 17) menos uno (17,); (b) un (100, 17) por uno (100,). ¿Funciona el (b)? ¿Por qué? (Pista: reglas del broadcasting.)

**E4.** Con `retornos = np.array([44.8, 499.9, 168.5, 499.7, 296.0, 43.2])`, cuenta con una máscara cuántos están **entre** 100 y 400.

**E5.** Explica por qué este código da un resultado inesperado y arréglalo:

```python
mejores = pesos[0:5]
pesos[0:5] = 0      # seguimos entrenando...
print(mejores)      # ¿los mejores pesos?
```

**E6.** Usa `evaluar_en_paralelo` para medir qué porcentaje de las veces se cae la política con ruedecillas (8, 8). ¿Y con (12, 8)? (NB11, E2.)

**E7.** Añade a `test_palo.py` un test `test_viento_cero_y_derecho_no_se_mueve` que compruebe que, sin viento (`PaloDeEscoba(viento_maximo=0)`) y con el palo **perfectamente derecho**, no se mueve.
(Pista: tras el `reset`, pon `entorno.inclinacion = 0.0` y da un paso con empuje 0.)

**E8.** ¿Qué tres cosas **nunca** deberían ir en un repositorio de Git? ¿Cómo se evita que vayan?


<details>
<summary>▶ Solución E1</summary>

```python
t = np.arange(20).reshape(4, 5)
print(t[:, 2])      # [ 2  7 12 17]
print(t[-1])        # [15 16 17 18 19]
```
</details>

<details>
<summary>▶ Solución E2</summary>

```python
print(resultados.std(axis=1).round(1))
```

Una desviación por política (`axis=1`: se resumen las semillas de cada fila). La más irregular, con mucha diferencia, es "solo inclinación" (unos 152): a veces aguanta y a veces cae enseguida. "A mano" tiene
desviación **0**: siempre da lo mismo.
</details>

<details>
<summary>▶ Solución E3</summary>

(a) (100, 17) − (17,): desde la derecha, 17 y 17 encajan → resultado **(100, 17)** (a cada fila se le resta el vector). (b) (100, 17) × (100,): desde la derecha, **17 contra 100**: ni iguales ni 1 → **error**. Para
multiplicar cada **fila** por un número distinto, el vector tendría que tener forma (100, 1): `v.reshape(-1, 1)`. Entonces, desde la derecha, 17 contra 1 (vale) y 100 contra 100 (vale).
</details>

<details>
<summary>▶ Solución E4</summary>

```python
retornos = np.array([44.8, 499.9, 168.5, 499.7, 296.0, 43.2])
print(((retornos > 100) & (retornos < 400)).sum())     # 2
```

Los paréntesis alrededor de cada comparación son obligatorios con `&`.
</details>

<details>
<summary>▶ Solución E5</summary>

`pesos[0:5]` es una **vista**: `mejores` y `pesos` comparten los datos. Al poner `pesos[0:5] = 0`, `mejores` también se llena de ceros. Arreglo: `mejores = pesos[0:5].copy()`.
</details>

<details>
<summary>▶ Solución E6</summary>

```python
for k, d in [(8, 8), (12, 8)]:
    r = evaluar_en_paralelo(k, d)
    print(k, d, f"{(r < 490).mean():.1%}")
```

Con (8, 8) se cae el **100 %** de las veces (la ruedecilla no vence a la "gravedad" de 10); con (12, 8), el **0 %**: ni una sola caída en mil episodios. Con mil episodios, la frontera del NB11 se ve con toda claridad.
</details>

<details>
<summary>▶ Solución E7</summary>

Añade al texto de `tests`:

```python
def test_viento_cero_y_derecho_no_se_mueve():
    entorno = PaloDeEscoba(viento_maximo=0)
    entorno.reset(seed=0)
    entorno.inclinacion = 0.0
    (inclinacion, velocidad), r, terminado, truncado, info = entorno.step(0)
    assert inclinacion == 0.0
    assert velocidad == 0.0
```

Sin viento, sin empuje y sin inclinación, ninguna fuerza actúa: el palo se queda quieto (la inercia del NB02). Vuelve a escribir el fichero y ejecuta `ejecutar_tests()`: debería decir `6 passed`.
</details>

<details>
<summary>▶ Solución E8</summary>

1. **Contraseñas, claves de APIs y datos privados** (lo más grave: aunque se borren, quedan en el historial).
2. **Entornos virtuales** (`venv/`): enormes, y solo sirven en tu ordenador; se sube el `requirements.txt`.
3. **Ficheros generados** que se pueden volver a crear (cachés, `__pycache__/`, resultados temporales, carpetas de prácticas).

Se evita con un fichero **`.gitignore`** con sus patrones, y revisando `git status` antes de cada commit.
</details>


## 16 · 🛠 Práctica en MuJoCo: ¿cómo sabes que tu simulador no miente?

Llevas desde el NB00 fiándote de MuJoCo: si dice que el palo cae en 0,68 s, te lo crees. Pero un ingeniero no se fía: **comprueba**. Y la forma
profesional de comprobar es la de hoy: **tests** que el ordenador ejecuta solo y que gritan si algo cambia.

¿Qué se puede comprobar de un simulador? Que cumpla cosas que **sabemos seguras** por la física o por cómo está hecho:

1. **Determinismo** (NB24): mismo comienzo y mismas órdenes → exactamente el mismo resultado.
2. **Caída libre** (NB02, NB04b): una pelota que cae debe seguir la fórmula **½·g·t²**.
3. **Conservación de la energía**: un péndulo **sin rozamiento** no puede ganar ni perder energía (si la gana, el simulador "se la inventa").
4. **El periodo del péndulo**: un péndulo de largo L oscila con un periodo de **2π·√(L/g)** (una fórmula clásica que verás en la Parte 5).
5. **Una política conocida**: el PD del NB23 sostiene el palo de escoba de MuJoCo 10 segundos.

Primero exploraremos la energía con NumPy, y después lo convertiremos todo en un fichero de tests para **pytest**, y estropearemos el modelo a
propósito para ver cómo los tests lo cazan.


### Paso 1 · Un péndulo que mide su energía

Un péndulo sencillo: una bola de 1 kg colgada de una varilla de 1 m (casi sin masa) en una bisagra. Dos novedades en el plano:

- `<flag energy="enable"/>`: le pide a MuJoCo que **calcule la energía** en cada paso. La deja en **`datos.energy`**, dos números: la energía
  **potencial** (la de la altura) y la **cinética** (la del movimiento). Su suma es la **energía total**.
- `integrator="{integrador}"` y `damping="{rozamiento}"` son huecos (NB20): el **integrador** es la receta con la que MuJoCo avanza cada paso
  (la "cadena de oro" del NB02 es una de ellas, llamada `Euler`; hay otras más precisas, como `RK4`). Lo guardaremos en un fichero (NB26).


In [30]:
from pathlib import Path
import mujoco
import numpy as np

PENDULO = '''<mujoco model="pendulo">
  <option timestep="0.001" integrator="{integrador}">
    <flag energy="enable"/>
  </option>
  <worldbody>
    <light pos="0 0 3"/>
    <body name="bola" pos="0 0 1.5">
      <joint name="bisagra" type="hinge" axis="0 1 0" damping="{rozamiento}"/>
      <geom type="capsule" fromto="0 0 0 0 0 -1" size="0.005" mass="0.001"/>
      <geom type="sphere" pos="0 0 -1" size="0.03" mass="1"/>
    </body>
  </worldbody>
</mujoco>'''

def simular_pendulo(integrador="Euler", rozamiento=0, segundos=10, grados=10):
    modelo = mujoco.MjModel.from_xml_string(PENDULO.format(integrador=integrador, rozamiento=rozamiento))
    datos = mujoco.MjData(modelo)
    datos.qpos[0] = np.radians(grados)
    mujoco.mj_forward(modelo, datos)
    n = round(segundos / modelo.opt.timestep)
    energia, angulo = np.empty(n + 1), np.empty(n + 1)        # arrays reservados de antemano
    energia[0], angulo[0] = datos.energy.sum(), datos.qpos[0]
    for i in range(1, n + 1):
        mujoco.mj_step(modelo, datos)
        energia[i], angulo[i] = datos.energy.sum(), datos.qpos[0]
    return energia, angulo

(En vez de listas con `append`, reservamos los arrays de golpe con `np.empty` y los rellenamos por índice: es lo habitual cuando sabes de antemano
cuántos datos habrá.) Fíjate en `PENDULO.format(...)`: es el **hermano** de las f-strings, que rellena los huecos `{ }` **más tarde**, cuando lo llamas.
Lo necesitamos porque queremos guardar la plantilla con los huecos sin rellenar.

### Paso 2 · ¿Se conserva la energía?

Soltamos el péndulo desde 10° y simulamos 10 segundos (10.000 pasos), con tres combinaciones: Euler sin rozamiento, RK4 sin rozamiento, y Euler
**con** rozamiento. Con NumPy (apartados 5-6) medimos cuánto ha cambiado la energía. Para que la cifra signifique algo, la comparamos con la energía
del **balanceo** (la que tiene el péndulo por estar levantado 10°: `m·g·L·(1 − cos 10°)`, unos 0,149 julios):


In [31]:
energia_balanceo = 1 * 9.81 * 1 * (1 - np.cos(np.radians(10)))
print(f"Energía del balanceo: {energia_balanceo:.3f} J\n")

for integrador, rozamiento in [("Euler", 0), ("RK4", 0), ("Euler", 0.05)]:
    energia, angulo = simular_pendulo(integrador, rozamiento)
    cambio = energia[-1] - energia[0]
    print(f"{integrador:>5}, rozamiento {rozamiento:<4}: cambio de energía {cambio:+.2e} J "
          f"= {cambio / energia_balanceo:+.4%} del balanceo")

Energía del balanceo: 0.149 J



Euler, rozamiento 0   : cambio de energía +7.34e-05 J = +0.0493% del balanceo


  RK4, rozamiento 0   : cambio de energía -3.57e-12 J = -0.0000% del balanceo


Euler, rozamiento 0.05: cambio de energía -5.88e-02 J = -39.4338% del balanceo


Léelo con calma, porque dice mucho:

- **Euler sin rozamiento**: la energía cambia en unas **siete cienmilésimas** de julio: un **0,05 %** del balanceo en 10 s. Casi nada, pero no cero:
  la cadena de oro con pasos de 1 ms comete un error diminuto en cada paso.
- **RK4 sin rozamiento**: el cambio es de **unas pocas billonésimas** de julio: prácticamente perfecto (el ruido de los decimales del NB06). RK4 es
  más preciso... y más lento (hace cuatro cálculos por paso en vez de uno).
- **Euler con rozamiento**: pierde **0,059 J, un 40 %** del balanceo. Correcto: el rozamiento **debe** gastar energía.

Así que un buen test de energía no puede pedir "cambio == 0" (fallaría con Euler). Tiene que pedir "cambio **pequeño** comparado con el balanceo",
por ejemplo menos del 1 %. Elegir la **tolerancia** de un test es parte del oficio.


### Paso 3 · El periodo, con máscaras de NumPy

El periodo es el tiempo de una oscilación completa. Lo medimos con NumPy **sin bucles**: el péndulo pasa por la vertical "subiendo" cuando el ángulo
es negativo en un paso y positivo (o cero) en el siguiente. Eso es una **máscara** (apartado 5) comparando el array consigo mismo **desplazado** un
paso, y `np.nonzero` nos da en qué pasos ocurre:


In [32]:
energia, angulo = simular_pendulo("RK4")
cruces = np.nonzero((angulo[:-1] < 0) & (angulo[1:] >= 0))[0]    # pasos donde cruza "subiendo"
tiempos_cruce = (cruces + 1) * 0.001
periodo = np.diff(tiempos_cruce).mean()                         # tiempo medio entre cruces

formula = 2 * np.pi * np.sqrt(1 / 9.81)
print(f"Cruces: {len(cruces)} | periodo medido: {periodo:.4f} s | fórmula: {formula:.4f} s "
      f"| diferencia: {(periodo - formula) / formula:+.2%}")

Cruces: 5 | periodo medido: 2.0100 s | fórmula: 2.0061 s | diferencia: +0.20%


(`angulo[:-1]` es "todos menos el último" y `angulo[1:]`, "todos menos el primero": puestos uno encima del otro, cada posición compara un paso con
el siguiente. Y `np.diff` resta cada elemento del siguiente.)

MuJoCo: **2,010 s**. Fórmula: **2,006 s**. Una diferencia del **0,2 %**, y ni siquiera es un error de MuJoCo: la fórmula es para oscilaciones
**pequeñitas**, y con 10° el periodo real es justo un 0,2 % más largo (la física fina, en la Parte 5). El simulador reproduce una ley de la física
de hace casi 400 años (Galileo, otra vez).


### Paso 4 · El fichero de tests

Ahora, lo profesional. En una carpeta propia (`practica_mujoco/nb27_tests`) guardamos el **plano del péndulo** como fichero (NB26) y un fichero
`test_simulacion.py` con **seis tests**, uno por idea. Los tests leen el plano del fichero que está **a su lado**: `Path(__file__).parent` es "la
carpeta de este fichero".


In [33]:
carpeta_tests = Path("practica_mujoco") / "nb27_tests"
carpeta_tests.mkdir(parents=True, exist_ok=True)
plano_pendulo = PENDULO.format(integrador="Euler", rozamiento=0)
(carpeta_tests / "pendulo.xml").write_text(plano_pendulo, encoding="utf-8")
print("Plano guardado en", carpeta_tests / "pendulo.xml")

Plano guardado en practica_mujoco/nb27_tests/pendulo.xml


In [34]:
tests_mujoco = '''# test_simulacion.py: tests de MuJoCo (práctica del NB27).
from pathlib import Path

import mujoco
import numpy as np
import pytest

AQUI = Path(__file__).parent
RAIZ = AQUI.parent.parent                       # la carpeta notebooks/


def cargar(ruta):
    modelo = mujoco.MjModel.from_xml_path(str(ruta))
    datos = mujoco.MjData(modelo)
    mujoco.mj_forward(modelo, datos)
    return modelo, datos


def test_determinismo_humanoide():
    import gymnasium
    ruta = Path(gymnasium.__file__).parent / "envs" / "mujoco" / "assets" / "humanoid.xml"
    finales = []
    for _ in range(2):
        modelo, datos = cargar(ruta)
        azar = np.random.default_rng(7)         # misma semilla las dos veces
        for _ in range(300):
            datos.ctrl[:] = azar.uniform(-0.4, 0.4, size=modelo.nu)
            mujoco.mj_step(modelo, datos)
        finales.append(datos.qpos.copy())
    np.testing.assert_array_equal(finales[0], finales[1])     # iguales, número por número


def test_caida_libre():
    modelo = mujoco.MjModel.from_xml_string(
        '<mujoco><option timestep="0.001" gravity="0 0 -10"/><worldbody>'
        '<body pos="0 0 2"><freejoint/><geom type="sphere" size="0.05"/></body>'
        '</worldbody></mujoco>')
    datos = mujoco.MjData(modelo)
    for _ in range(500):                        # 0,5 s
        mujoco.mj_step(modelo, datos)
    assert datos.qpos[2] == pytest.approx(2 - 0.5 * 10 * 0.5 ** 2, abs=0.005)


def oscilar(segundos=10, grados=10):
    modelo, datos = cargar(AQUI / "pendulo.xml")
    datos.qpos[0] = np.radians(grados)
    mujoco.mj_forward(modelo, datos)
    energia0 = datos.energy.sum()
    angulos = [datos.qpos[0]]
    for _ in range(round(segundos / modelo.opt.timestep)):
        mujoco.mj_step(modelo, datos)
        angulos.append(datos.qpos[0])
    return energia0, datos.energy.sum(), np.array(angulos), modelo.opt.timestep


def test_energia_se_conserva_sin_rozamiento():
    energia0, energia1, _, _ = oscilar()
    balanceo = 1 * 9.81 * 1 * (1 - np.cos(np.radians(10)))
    assert abs(energia1 - energia0) < 0.01 * balanceo          # menos del 1 %


def test_periodo_del_pendulo():
    _, _, angulos, dt = oscilar()
    cruces = np.nonzero((angulos[:-1] < 0) & (angulos[1:] >= 0))[0]
    periodo = np.diff(cruces).mean() * dt
    assert periodo == pytest.approx(2 * np.pi * np.sqrt(1 / 9.81), rel=0.01)


def test_palo_de_escoba_pd_aguanta_10_segundos():
    modelo, datos = cargar(RAIZ / "robots" / "palo_escoba.xml")
    datos.qpos[1] = np.radians(5)
    for _ in range(1000):
        x, angulo = datos.qpos
        vx, va = datos.qvel
        datos.ctrl[0] = np.clip(3 * angulo + 0.8 * va + 0.1 * x + 0.2 * vx, -1, 1)
        mujoco.mj_step(modelo, datos)
        assert abs(angulo) < 0.785


def test_plano_roto_da_valueerror():
    with pytest.raises(ValueError):
        mujoco.MjModel.from_xml_string("<mujoco><worldbody><body></worldbody></mujoco>")
'''
(carpeta_tests / "test_simulacion.py").write_text(tests_mujoco, encoding="utf-8")
print("Tests escritos")

Tests escritos


Repásalos: cada uno comprueba **una** idea y su nombre dice **cuál**. Tres herramientas de hoy:

- **`np.testing.assert_array_equal`**: dos arrays **idénticos** (para el determinismo queremos igualdad exacta, no "parecidos").
- **`pytest.approx`**, con `abs=` (margen absoluto) o `rel=` (margen relativo: `rel=0.01` es "dentro del 1 %").
- **`pytest.raises(ValueError)`**: un test que **pasa si salta** ese error (el plano roto del NB22 **debe** quejarse).

Ejecutémoslos, con `subprocess` como en el apartado 11:


In [35]:
import subprocess, sys

def pasar_tests():
    resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"],
                               cwd=carpeta_tests, capture_output=True, text=True)
    print(resultado.stdout.strip()[-900:])

pasar_tests()

......                                                                   [100%]
6 passed in 0.43s


**`6 passed`**: MuJoCo es determinista, cumple la caída libre, conserva la energía, oscila con el periodo de la fórmula, el PD sostiene el palo y
los planos rotos se quejan. Ahora tienes una **red de seguridad**: si mañana actualizas MuJoCo, cambias un plano o tocas un controlador, ejecutas
`pytest` y en unos segundos sabes si algo se ha roto.


### Paso 5 · Los tests cazan un fallo

Como en el apartado 11, estropeamos algo **a propósito**. Imagina que alguien edita `pendulo.xml` para "que quede más realista" y le pone rozamiento
a la bisagra (`damping="0.05"`), sin avisar. Volvemos a pasar los tests:


In [36]:
ruta_plano = carpeta_tests / "pendulo.xml"
original = ruta_plano.read_text(encoding="utf-8")
_ = ruta_plano.write_text(original.replace('damping="0"', 'damping="0.05"'), encoding="utf-8")

pasar_tests()

_ = ruta_plano.write_text(original, encoding="utf-8")      # lo dejamos como estaba

============================== FAILURES ===================================
___________________ test_energia_se_conserva_sin_rozamiento ____________________

    def test_energia_se_conserva_sin_rozamiento():
        energia0, energia1, _, _ = oscilar()
        balanceo = 1 * 9.81 * 1 * (1 - np.cos(np.radians(10)))
>       assert abs(energia1 - energia0) < 0.01 * balanceo          # menos del 1 %
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E       assert np.float64(0.058770461398892415) < (0.01 * np.float64(0.14903594295023934))
E        +  where np.float64(0.058770461398892415) = abs((np.float64(5.0051499995228195) - np.float64(5.063920460921712)))

test_simulacion.py:59: AssertionError
=========================== short test summary info ============================
FAILED test_simulacion.py::test_energia_se_conserva_sin_rozamiento - assert n...
1 failed, 5 passed in 0.44s


**`1 failed, 5 passed`**: ha fallado justo `test_energia_se_conserva_sin_rozamiento`, y pytest enseña los números: la energía ha cambiado mucho
más del 1 % permitido. El test del **periodo**, en cambio, sigue pasando: un rozamiento pequeño apenas cambia el ritmo del péndulo, solo lo va
frenando. Por eso conviene tener **varios** tests que miren cosas **distintas**: cada uno caza fallos diferentes.

Una última vez, para comprobar que lo hemos dejado bien:


In [37]:
pasar_tests()

......                                                                   [100%]
6 passed in 0.42s


### Tus retos

**Reto 1.** En el Paso 2, cambia el paso de tiempo del péndulo Euler a 0,01 s (diez veces más grande; pista: `PENDULO.replace('0.001', '0.01')`
antes de formatear, dentro de una copia de `simular_pendulo`). ¿Cuánto cambia ahora la energía en 10 s? ¿Pasaría el test del 1 %?

**Reto 2.** Añade al fichero de tests uno nuevo, `test_luna`: con gravedad `0 0 -1.62`, la pelota del test de caída libre debe estar, a los 0,5 s, a
**2 − ½·1,62·0,5²** metros. Escríbelo y pasa los tests.

**Reto 3 (para pensar).** El test `test_determinismo_humanoide` compara **dos ejecuciones en el mismo ordenador**. ¿Crees que dos ordenadores distintos
(la Pi y un portátil) darían exactamente los mismos números? ¿Qué tolerancia pondrías si quisieras comparar con unos resultados guardados en otro ordenador?

<details>
<summary>▶ Solución Reto 1</summary>

```python
def simular_pendulo_paso(paso, segundos=10, grados=10):
    plano = PENDULO.replace('timestep="0.001"', f'timestep="{paso}"').format(integrador="Euler", rozamiento=0)
    modelo = mujoco.MjModel.from_xml_string(plano)
    datos = mujoco.MjData(modelo)
    datos.qpos[0] = np.radians(grados)
    mujoco.mj_forward(modelo, datos)
    e0 = datos.energy.sum()
    for _ in range(round(segundos / paso)):
        mujoco.mj_step(modelo, datos)
    return datos.energy.sum() - e0

cambio = simular_pendulo_paso(0.01)
print(f"{cambio:+.2e} J = {cambio / energia_balanceo:+.2%} del balanceo")
```

Con pasos 10 veces más grandes, el error de Euler se hace unas 10 veces más grande: **+8,5·10⁻⁴ J, un 0,57 %** del balanceo en 10 s (frente al 0,05 % con pasos de 1 ms). Todavía pasaría
el test del 1 %, pero por poco; y en una simulación más larga, no. Es la regla del NB22: el paso tiene que ser pequeño comparado con lo rápido que pasan
las cosas. (Y es otra razón para tener tests: si alguien "acelera" la simulación subiendo el paso, el test de energía avisará.)
</details>

<details>
<summary>▶ Solución Reto 2</summary>

Añade al texto de `tests_mujoco` (y vuelve a ejecutar esa celda y la de `pasar_tests()`):

```python
def test_luna():
    modelo = mujoco.MjModel.from_xml_string(
        '<mujoco><option timestep="0.001" gravity="0 0 -1.62"/><worldbody>'
        '<body pos="0 0 2"><freejoint/><geom type="sphere" size="0.05"/></body>'
        '</worldbody></mujoco>')
    datos = mujoco.MjData(modelo)
    for _ in range(500):
        mujoco.mj_step(modelo, datos)
    assert datos.qpos[2] == pytest.approx(2 - 0.5 * 1.62 * 0.5 ** 2, abs=0.005)
```

Resultado: **`7 passed`**. (La pelota está a unos 1,80 m: en la Luna, en medio segundo, solo cae 20 cm.)
</details>

<details>
<summary>▶ Solución Reto 3</summary>

Probablemente **no** idénticos. MuJoCo es determinista en un mismo ordenador y con la misma versión, pero distintos procesadores (ARM en la Pi, x86 en
muchos portátiles) o distintas versiones de las bibliotecas pueden redondear los decimales de forma ligeramente distinta, y en un sistema caótico como
un humanoide cayendo esas diferencias diminutas crecen. Para comparar con resultados de **otro** ordenador usarías `np.testing.assert_allclose` con una
tolerancia, y mejor sobre una simulación **corta** o sobre algo estable (como el péndulo), no sobre 300 pasos de caída caótica.
</details>

### Qué has aprendido de MuJoCo hoy

- Con `<flag energy="enable"/>`, MuJoCo calcula en **`datos.energy`** la energía potencial y cinética.
- El **integrador** (`Euler`, `RK4`...) es la receta de cada paso: RK4 conserva la energía casi perfectamente; Euler comete un error pequeño que crece
  con el paso de tiempo. El rozamiento (`damping`) **debe** gastar energía.
- MuJoCo reproduce leyes de la física (caída libre, periodo del péndulo) con errores de décimas de porcentaje, y es **determinista**.
- Un fichero de **tests con pytest** (`assert_array_equal`, `pytest.approx`, `pytest.raises`) es tu red de seguridad: caza al instante un cambio
  indebido en un plano o en el código.

Con esto cierras el bloque de Python **sabiendo simular, envolver, guardar y verificar** robots en MuJoCo. En la práctica del NB28 empezarás a usar el
**azar** de forma controlada: estados iniciales aleatorios y la distribución de los tiempos de caída del palo de escoba.


## 17 · Posdata: se acaba el bloque de Python

Si algo no ha quedado claro, dime el **apartado** y la **frase exacta** y lo reescribo.

**Enhorabuena: has terminado el bloque "Python de verdad".** Mira todo lo que sabes ahora: texto y f-strings, colecciones y la trampa del alias, `while` y excepciones, depurar, funciones a fondo
(cierres, decoradores, generadores), clases y herencia, tu propio entorno de Gymnasium, ficheros y formatos, módulos, scripts, la terminal, `pip`, NumPy a fondo con ejes y broadcasting, entornos
vectorizados, tests y Git; y, en las prácticas, cómo generar, recorrer, romper, envolver, guardar y verificar simulaciones de MuJoCo. **Es, sin exagerar, el Python que se usa en un trabajo de verdad.** A partir de aquí, el código de las bibliotecas profesionales dejará de parecer magia.

En la **Parte 4** vuelve la robótica, a lo grande: el **aprendizaje por refuerzo de verdad**. Empezaremos por la **probabilidad desde cero** (la campana de Gauss que ya ha asomado hoy), para construir
políticas que **exploran** con azar; y con ella, el primer algoritmo que aprende **solo con recompensas**, sin maestro: **REINFORCE**. Después vendrán PyTorch, actor-crítico y PPO, el algoritmo con el que
se entrenan los humanoides. Y lo entrenaremos sobre **tu** palo de escoba, el entorno de Gymnasium del NB25.
